# EDA to Machine Learning ? Python Methods Notebook

This notebook covers the key methods from the ML cheat sheet.

## Topics included
- Read data
- Separate numerical and categorical columns
- Data quick checks
- Missing values
- Data cleaning
- Categorical analysis
- Numerical analysis
- Outlier detection
- Bivariate and multivariate analysis
- Encoding
- Scaling
- Feature engineering
- Feature selection
- Dimensionality reduction
- Train/test split and cross-validation
- Regression algorithms
- Classification algorithms
- Unsupervised learning
- Ensemble learning
- Hyperparameter tuning
- Model evaluation
- Imbalanced data handling
- Deployment and monitoring

In [ ]:
# 1. Install packages
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn xgboost lightgbm catboost imbalanced-learn category-encoders shap optuna mlflow fastapi joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
print('Packages ready')


In [ ]:
# 2. Read the data

df_csv = pd.read_csv('data.csv')
df_excel = pd.read_excel('data.xlsx')
df_json = pd.read_json('data.json')
df_parquet = pd.read_parquet('data.parquet')

print(type(df_csv).__name__, df_csv.shape)
print(type(df_excel).__name__, df_excel.shape)
print(type(df_json).__name__, df_json.shape)
print(type(df_parquet).__name__, df_parquet.shape)


In [ ]:
# 3. Separate numerical and categorical columns

df = pd.DataFrame({
    'age': [25, 30, 41, 52],
    'salary': [50000, 60000, 90000, 120000],
    'city': ['New York', 'London', 'Paris', 'Tokyo'],
    'gender': ['M', 'F', 'F', 'M'],
    'is_active': [True, False, True, False],
})

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns

print('Numeric columns:', list(num_cols))
print('Categorical columns:', list(cat_cols))


In [ ]:
# 4. Data quick checks

df = pd.DataFrame({
    'age': [25, 30, 41, 52, 29],
    'salary': [50000, 60000, 90000, 120000, 55000],
    'city': ['A', 'B', 'A', 'C', 'B'],
})

print('Shape:', df.shape)
print('Size:', df.size)
print('Rows:', len(df))
print('Head:\n', df.head())
print('Tail:\n', df.tail())
print('Sample:\n', df.sample(3))
print('Info:\n')
df.info()
print('Summary:\n', df.describe().T)
print('Dtypes:\n', df.dtypes)
print('Unique counts:\n', df.nunique())
print('Value counts:\n', df['city'].value_counts())
print('Duplicate count:', df.duplicated().sum())
print('Memory usage:\n', df.memory_usage(deep=True))


In [ ]:
# 5. Missing value analysis and treatment

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

df = pd.DataFrame({
    'age': [25, None, 41, 52, None],
    'salary': [50000, 60000, None, 120000, 55000],
    'city': ['A', 'B', None, 'C', 'B'],
})

print('Missing values:\n', df.isnull().sum())
print('Missing %:\n', df.isna().mean() * 100)

# Pandas fill methods
df['age'] = df['age'].fillna(df['age'].median())
df['salary'] = df['salary'].fillna(df['salary'].mean())
df['city'] = df['city'].fillna(df['city'].mode()[0])
df_ffill = df.ffill()
df_bfill = df.bfill()

print('After fillna:\n', df)
print('FFill:\n', df_ffill)
print('BFill:\n', df_bfill)

# Scikit-learn imputation
num_cols = df.select_dtypes(include=np.number).columns
simple_imputer = SimpleImputer(strategy='median')
df_simple = pd.DataFrame(simple_imputer.fit_transform(df[num_cols]), columns=num_cols)
print('SimpleImputer:\n', df_simple)

knn_imputer = KNNImputer(n_neighbors=2)
df_knn = pd.DataFrame(knn_imputer.fit_transform(df[num_cols]), columns=num_cols)
print('KNNImputer:\n', df_knn)


In [ ]:
# 6. Data quality checks and cleaning

df = pd.DataFrame({
    'name': [' Alice ', 'BOB', 'charlie'],
    'city': [' New York ', 'new york', 'Paris'],
    'salary': ['50000', '60000', 'bad'],
    'date': ['2023-01-01', 'invalid', '2023-03-15']
})

df['name'] = df['name'].str.strip().str.lower()
df['city'] = df['city'].str.strip().str.lower()
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.drop_duplicates()
df = df.rename(columns={'name': 'employee_name'})
print(df)


In [ ]:
# 7. Categorical column analysis

df = pd.DataFrame({
    'city': ['New York', 'London', 'Paris', 'New York', 'London'],
    'gender': ['M', 'F', 'F', 'M', 'F'],
    'purchased': [1, 0, 1, 1, 0],
})

print(df['city'].value_counts())
print(df['city'].value_counts(normalize=True))
print(pd.crosstab(df['city'], df['gender']))
print(df.groupby('city').size())
print(df.groupby('city')['purchased'].mean())

sns.countplot(data=df, x='city')
plt.show()


In [ ]:
# 8. Numerical column analysis

df = pd.DataFrame({'age': [22, 24, 26, 30, 35, 40, 100]})

print('mean:', df['age'].mean())
print('median:', df['age'].median())
print('mode:', df['age'].mode().tolist())
print('min/max:', df['age'].min(), df['age'].max())
print('std:', df['age'].std())
print('skew:', df['age'].skew())
print('kurtosis:', df['age'].kurt())

sns.histplot(data=df, x='age', kde=True)
plt.show()
sns.boxplot(data=df, x='age')
plt.show()

from scipy import stats
stats.probplot(df['age'].dropna(), dist='norm', plot=plt)
plt.show()


In [ ]:
# 9. Outlier detection and treatment

df = pd.DataFrame({'age': [20, 22, 24, 23, 26, 25, 90, 28, 29, 30]})

Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['age'] < lower) | (df['age'] > upper)]
print('Outliers:', outliers['age'].tolist())

z = np.abs(zscore(df['age']))
print('Z-score outliers:', df[z > 3]['age'].tolist())

df['age_clipped'] = df['age'].clip(lower=lower, upper=upper)
print(df)


In [ ]:
# 10. Bivariate and multivariate analysis

df = pd.DataFrame({
    'age': [22, 25, 30, 35, 40, 45],
    'salary': [40000, 50000, 65000, 80000, 90000, 110000],
    'gender': ['M', 'F', 'F', 'M', 'F', 'M'],
    'purchased': [0, 1, 1, 0, 1, 1],
})

print(df[['age', 'salary']].corr(method='pearson'))
sns.scatterplot(data=df, x='age', y='salary')
plt.show()
sns.boxplot(data=df, x='gender', y='salary')
plt.show()
pd.crosstab(df['gender'], df['purchased'])
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()


In [ ]:
# 11. Encoding categorical features

import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

# Small example data: city is nominal; size has a meaningful order.
df = pd.DataFrame({
    'city': ['Paris', 'London', 'Paris', 'Tokyo', 'London'],
    'size': ['low', 'medium', 'high', 'medium', 'low'],
    'purchased': [1, 0, 1, 1, 0],
})

# A) Manual mappings: best for a known, intentional mapping.
city_map = {'Paris': 1, 'London': 2, 'Tokyo': 3}
df['city_map'] = df['city'].map(city_map)  # unmapped values become NaN
df['city_replace'] = df['city'].replace(city_map)  # keeps values not in the map

# B) Pandas integer codes: compact, but the integers imply no real order.
df['city_category_codes'] = df['city'].astype('category').cat.codes
city_codes, city_levels = pd.factorize(df['city'])
df['city_factorized'] = city_codes
print('factorize levels:', list(city_levels))

# C) Count and frequency encoding: useful for high-cardinality categories.
city_counts = df['city'].value_counts()
city_frequency = df['city'].value_counts(normalize=True)
df['city_count'] = df['city'].map(city_counts)
df['city_frequency'] = df['city'].map(city_frequency)

# D) One-hot / dummy encoding: nominal categories without invented order.
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
city_ohe = ohe.fit_transform(df[['city']])
city_ohe_df = pd.DataFrame(
    city_ohe, columns=ohe.get_feature_names_out(['city']), index=df.index
)
dummies = pd.get_dummies(df['city'], prefix='city', drop_first=True, dtype=int)

# E) Ordinal encoding: specify the real order explicitly.
size_order = ['low', 'medium', 'high']
ordinal = OrdinalEncoder(
    categories=[size_order],
    handle_unknown='use_encoded_value',
    unknown_value=-1,
)
df['size_ordinal'] = ordinal.fit_transform(df[['size']]).ravel()

# F) LabelEncoder is mainly for a 1D target y, not nominal input features.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['purchased'])
print('target classes:', list(label_encoder.classes_))

print(df)
print('One-hot columns:\n', city_ohe_df)
print('Dummy columns:\n', dummies)


## Categorical encoding methods: when to use them

| Method | Best for | Important note |
|---|---|---|
| `Series.map(dict)` / `Series.replace(dict)` | Small, fixed mappings | `map` turns unknown values into missing; `replace` leaves them unchanged. |
| `astype('category').cat.codes` / `pd.factorize` | Quick integer codes for exploration | Codes are arbitrary and can accidentally imply order; handle missing/unknown categories. |
| `OneHotEncoder` / `pd.get_dummies` | Nominal categories with low/moderate cardinality | Use `handle_unknown='ignore'` for unseen inference categories; fit encoders on training data. |
| `OrdinalEncoder` | Categories with a genuine order | Provide the order explicitly, such as low < medium < high. |
| Count / frequency encoding | High-cardinality categories | Learn counts from training data only; define an unknown-category fallback. |
| Label encoding | Usually target labels (`y`) | Avoid using arbitrary label codes for nominal input features with linear/distance-based models. |
| Target / WOE / leave-one-out / CatBoost encoding | Supervised, high-cardinality features | These use `y`; use cross-fitting/out-of-fold encoding inside training folds to prevent target leakage. |
| Binary / hashing encoding | Very high-cardinality features | Reduces width; hashing can cause collisions and is not directly interpretable. |

## Optional `category_encoders` methods

Install only if needed: `%pip install category-encoders`. The following cell demonstrates its unsupervised encoders. Supervised encoders are listed there too; use them in a leakage-safe cross-validation workflow, not by encoding the full dataset before splitting.


In [ ]:
# Optional: install category-encoders with %pip install category-encoders
# BinaryEncoder, HashingEncoder and CountEncoder do not use the target.
try:
    import category_encoders as ce
except ImportError as exc:
    raise ImportError("Install the optional package with: %pip install category-encoders") from exc

X_cat = df[['city']].copy()
y = df['purchased']

advanced_unsupervised = {
    'binary': ce.BinaryEncoder(cols=['city']),
    'hashing': ce.HashingEncoder(cols=['city'], n_components=4),
    'count': ce.CountEncoder(cols=['city']),
}
for name, encoder in advanced_unsupervised.items():
    print(f'{name} encoding:\n', encoder.fit_transform(X_cat))

# Supervised encoder APIs (do not fit on the full dataset before splitting):
target_encoder = ce.TargetEncoder(cols=['city'])
woe_encoder = ce.WOEEncoder(cols=['city'])
leave_one_out_encoder = ce.LeaveOneOutEncoder(cols=['city'])
catboost_encoder = ce.CatBoostEncoder(cols=['city'])
print('Supervised encoders:', [
    type(encoder).__name__ for encoder in
    [target_encoder, woe_encoder, leave_one_out_encoder, catboost_encoder]
])

# For scikit-learn >= 1.3, TargetEncoder.fit_transform uses cross-fitting.
# If supported by your installed version, use it within each training fold:
# from sklearn.preprocessing import TargetEncoder
# target_encoder = TargetEncoder(cv=5, shuffle=True, random_state=42)
# X_target_encoded = target_encoder.fit_transform(X_cat, y)


In [ ]:
# 12. Feature scaling

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer

X = pd.DataFrame({
    'age': [20, 30, 40, 50],
    'salary': [30000, 60000, 90000, 120000],
})

std = StandardScaler().fit_transform(X)
mm = MinMaxScaler().fit_transform(X)
rob = RobustScaler().fit_transform(X)
pt = PowerTransformer(method='yeo-johnson').fit_transform(X)

print('StandardScaler:\n', std)
print('MinMaxScaler:\n', mm)
print('RobustScaler:\n', rob)
print('PowerTransformer:\n', pt)


In [ ]:
# 13. Feature transformation and engineering

df = pd.DataFrame({'age': [20, 30, 40, 50], 'salary': [50000, 60000, 70000, 90000], 'date': ['2020-01-01', '2021-06-01', '2022-07-15', '2023-09-10']})

df['log_salary'] = np.log1p(df['salary'])
df['age_sq'] = np.square(df['age'])
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['dayofweek'] = df['date'].dt.dayofweek

from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[['age', 'salary']])

print(df.head())
print(poly_features[:2])


In [ ]:
# 14. Feature selection and dimensionality reduction

from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

X = pd.DataFrame({
    'f1': [1, 2, 3, 4, 5, 6],
    'f2': [5, 6, 7, 8, 9, 10],
    'f3': [2, 2, 2, 3, 3, 4],
    'f4': [10, 9, 8, 7, 6, 5],
})
y = np.array([0, 0, 0, 1, 1, 1])

selector = SelectKBest(score_func=mutual_info_classif, k=2)
X_sel = selector.fit_transform(X, y)

rfe = RFE(estimator=LogisticRegression(max_iter=500), n_features_to_select=2)
X_rfe = rfe.fit_transform(X, y)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print('Selected shape:', X_sel.shape)
print('RFE shape:', X_rfe.shape)
print('PCA shape:', X_pca.shape)


In [ ]:
# 15. Train-test split and cross-validation

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LogisticRegression

X = np.arange(100).reshape(-1, 1)
y = np.array([0, 1] * 50)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', len(X_train), 'Test size:', len(X_test))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ts = TimeSeriesSplit(n_splits=5)

model = LogisticRegression(max_iter=500)
scores = cross_val_score(model, X, y, cv=skf)
print('Cross-val scores:', scores)


In [ ]:
# 16. Regression algorithms

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

X = np.array([[1], [2], [3], [4], [5], [6]])
y = np.array([2, 4, 6, 8, 10, 12], dtype=float)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(random_state=42),
    'KNeighborsRegressor': KNeighborsRegressor(n_neighbors=2),
    'SVR': SVR(kernel='rbf'),
}

for name, model in models.items():
    model.fit(X, y)
    pred = model.predict(X[:2])
    print(name, pred)


In [ ]:
# 17. Classification algorithms

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

X = np.array([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]])
y = np.array([0, 0, 0, 1, 1, 1])

models = {
    'LogisticRegression': LogisticRegression(max_iter=500),
    'KNeighborsClassifier': KNeighborsClassifier(n_neighbors=3),
    'GaussianNB': GaussianNB(),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'SVC': SVC(probability=True),
    'LinearSVC': LinearSVC(),
    'LinearDiscriminantAnalysis': LinearDiscriminantAnalysis(),
    'QuadraticDiscriminantAnalysis': QuadraticDiscriminantAnalysis()
}

for name, model in models.items():
    model.fit(X, y)
    print(name, model.predict(X[:2]))


In [ ]:
# 18. Unsupervised learning and anomaly detection

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score

X = np.array([[1, 2], [1, 4], [1, 0], [10, 2], [10, 4], [10, 0]])

kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(X)
print('KMeans labels:', labels)
print('Silhouette Score:', silhouette_score(X, labels))

agg = AgglomerativeClustering(n_clusters=2)
print('Agglomerative labels:', agg.fit_predict(X))

db = DBSCAN(eps=2, min_samples=2)
print('DBSCAN labels:', db.fit_predict(X))


In [ ]:
# 19. Ensemble learning

from sklearn.ensemble import BaggingClassifier, VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X = np.array([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]])
y = np.array([0, 0, 0, 1, 1, 1])

bag = BaggingClassifier(estimator=RandomForestClassifier(random_state=42), n_estimators=10, random_state=42)
bag.fit(X, y)
print('Bagging prediction:', bag.predict(X[:2]))

voting = VotingClassifier(estimators=[
    ('rf', RandomForestClassifier(random_state=42)),
    ('lr', LogisticRegression(max_iter=500)),
])
voting.fit(X, y)
print('Voting prediction:', voting.predict(X[:2]))


# 20. Hyperparameter tuning

Hyperparameters are settings chosen before/during model fitting (for example, tree depth). Keep a final test set untouched: tune only on training data with cross-validation, then evaluate the selected estimator once on the test set. When preprocessing is needed, tune a `Pipeline` and use parameter names such as `model__max_depth`.

| Method | When to use |
|---|---|
| `GridSearchCV` | Small, focused parameter grids; evaluates every combination. |
| `RandomizedSearchCV` | Larger spaces; samples a fixed number of combinations. |
| `HalvingGridSearchCV` / `HalvingRandomSearchCV` | Successive halving eliminates weak candidates early; experimental import is required. |
| Optuna / `BayesSearchCV` | Optional libraries for guided search in large or expensive spaces. |

Use a scoring metric that reflects the task (for example, `f1` for classification with class imbalance). `best_estimator_` is refit on all training data by default; `best_score_` is the mean cross-validation score, not a test score.


In [ ]:
# GridSearchCV, RandomizedSearchCV and successive-halving examples
from scipy.stats import randint
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

# Halving search is experimental in scikit-learn and needs this enable import.
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV

X, y = make_classification(
    n_samples=240,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
base_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# 1) Exhaustive search over a small, explicit grid.
param_grid = {
    'n_estimators': [20, 40],
    'max_depth': [None, 4],
    'min_samples_leaf': [1, 3],
}
grid_search = GridSearchCV(
    base_model, param_grid, scoring='f1', cv=cv, n_jobs=-1, refit=True
)
grid_search.fit(X_train, y_train)
print('Grid best params:', grid_search.best_params_)
print('Grid best CV F1:', grid_search.best_score_)
print('Grid test score:', grid_search.score(X_test, y_test))

# 2) Randomly sample combinations from distributions.
param_distributions = {
    'n_estimators': randint(20, 81),
    'max_depth': [None, 3, 5, 8],
    'min_samples_leaf': randint(1, 5),
}
random_search = RandomizedSearchCV(
    base_model,
    param_distributions,
    n_iter=8,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
random_search.fit(X_train, y_train)
print('Randomized best params:', random_search.best_params_)
print('Randomized best CV F1:', random_search.best_score_)
print('Randomized test score:', random_search.score(X_test, y_test))

# 3) Successive halving: start with more candidates and allocate more resources
# to the stronger candidates. These classes currently require experimental enablement.
halving_grid = HalvingGridSearchCV(
    base_model, param_grid, scoring='f1', cv=cv, factor=2, n_jobs=-1
)
halving_grid.fit(X_train, y_train)
print('Halving grid best params:', halving_grid.best_params_)

halving_random = HalvingRandomSearchCV(
    base_model,
    param_distributions,
    n_candidates=8,
    scoring='f1',
    cv=cv,
    random_state=42,
    factor=2,
    n_jobs=-1,
)
halving_random.fit(X_train, y_train)
print('Halving randomized best params:', halving_random.best_params_)


## Optional guided tuning: Optuna and BayesSearchCV

Install only the library you plan to use (`%pip install optuna` or `%pip install scikit-optimize`). These are templates; run them after defining `X_train`, `y_train`, and `cv` in the previous cell.

### Optuna
```python
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
    }
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring='f1').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(study.best_params, study.best_value)
```

### Bayesian search with scikit-optimize
```python
from skopt import BayesSearchCV
from skopt.space import Integer
from sklearn.ensemble import RandomForestClassifier

search = BayesSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {'n_estimators': Integer(50, 300), 'max_depth': Integer(2, 20)},
    n_iter=25, scoring='f1', cv=cv, random_state=42, n_jobs=-1,
)
search.fit(X_train, y_train)
print(search.best_params_, search.best_score_)
```


In [ ]:
# 20. Model evaluation

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

y_true = np.array([0, 1, 1, 0, 1, 0])
y_pred = np.array([0, 1, 0, 0, 1, 1])
y_score = np.array([0.1, 0.8, 0.6, 0.2, 0.9, 0.7])

print('Accuracy:', accuracy_score(y_true, y_pred))
print('Precision:', precision_score(y_true, y_pred, zero_division=0))
print('Recall:', recall_score(y_true, y_pred, zero_division=0))
print('F1:', f1_score(y_true, y_pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))
print('ROC AUC:', roc_auc_score(y_true, y_score))
fpr, tpr, _ = roc_curve(y_true, y_score)
plt.plot(fpr, tpr)
plt.title('ROC Curve')
plt.show()


In [ ]:
# 21. Imbalanced data handling

from imblearn.over_sampling import SMOTE

X = np.array([[1], [2], [3], [4], [100], [101], [102], [103]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print('Before:', np.bincount(y))
print('After:', np.bincount(y_res))


In [ ]:
# 22. Deployment and monitoring

import joblib
import pickle

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
X = np.array([[1, 2], [2, 3], [3, 4], [4, 5]])
y = np.array([0, 0, 1, 1])
model.fit(X, y)

joblib.dump(model, 'model.joblib')
loaded = joblib.load('model.joblib')
print('Loaded model prediction:', loaded.predict(X[:2]))

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved with pickle')


In [ ]:
# 23. End-to-end scikit-learn pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.DataFrame({
    'age': [25, 30, 32, 40, 45, 50],
    'salary': [50000, 55000, 60000, 70000, 90000, 100000],
    'city': ['NY', 'LA', 'NY', 'SF', 'LA', 'NY'],
    'target': [0, 0, 1, 1, 1, 1]
})

X = df.drop(columns=['target'])
y = df['target']

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])
model = Pipeline([('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model.fit(X_train, y_train)
preds = model.predict(X_test)
print(classification_report(y_test, preds))
